# IMF Palau 2025 Article IV — PDF-to-Dashboard AI Pipeline

**Data Analysis with AI | University coursework | Reproducible Google Colab notebook**

**Source:** International Monetary Fund (2026), *Republic of Palau: 2025 Article IV Consultation—Press Release; Staff Report; and Statement by the Executive Director*, IMF Country Report No. 26/22 (71-page PDF; published January 2026).  
**Purpose:** PDF processing → token-aware chunking → grounded LLM summaries → schema-controlled indicator extraction → seven-theme analysis → FAISS retrieval → automated evaluation → visual dashboard and optional prompt benchmark.

> **Academic integrity:** Run the notebook, personally check every numerical claim against the IMF report, record discrepancies, and write your own 5–6-page analysis. An AI-generated written report is explicitly prohibited by the brief. The notebook is a reproducible starting implementation, not a guarantee of marks.

**Requirements:** Google Colab runtime, uploaded assigned PDF, Google Gemini API key supplied securely at runtime, internet for installing packages and calling the model. API quotas/charges may apply. The notebook **does not contain a key** and **does not fabricate model results**.

**Important Palau interpretation:** The country uses the US dollar and has no domestic central-bank policy interest rate. For unavailable/inapplicable figures, the numeric schema uses `-1.0` and stores the explanation separately. The IMF's **commercial-bank** NPL and Tier 1 series exclude the National Development Bank of Palau (NDBP); do not conflate them. The 2023 Tier 1 risk-weighted-assets ratio is **0.08%** in Table 5; do not confuse with 1.25% Tier 1-to-assets.

**Execution notice:** API-generated results must be verified by the student. No grade can be guaranteed; outputs depend on live Gemini availability and account quotas. This version preserves raw/model-corrected indicators separately, checkpoints tasks, and provides a minimum of four source-derived charts even if optional judge calls have not finished.


## 0 · Setup and configuration
Run cells sequentially using **Runtime → Run all**. When prompted, upload the PDF and provide a Gemini API key; if Gemini is unavailable the notebook stops instead of silently fabricating outputs. The primary model is configured as `gemini-3.8-flash`, based on the error message returned by your API. API models may change: edit `GENERATOR_MODEL`/`JUDGE_MODEL` if your account does not offer the defaults. Confirm the model is available for your API key before running expensive summary/evaluation loops.
**For rate-limited keys:** Run the notebook one section at a time and allow quota resets, or use a billing-enabled project with an appropriate budget. Cache and checkpoints are saved in `/content/palau_imf_outputs`, which is temporary unless you copy it to Drive.

**Colab environment note:** The installation cell intentionally does not upgrade Pandas, NumPy, or Matplotlib because Colab includes them. If you have already run a version that upgraded Pandas, use **Runtime → Disconnect and delete runtime** once before running this updated notebook. This avoids mixed Pandas versions loaded in the same Python process.

**Quota-aware run:** the judge, RAG and optional benchmark cells checkpoint successful requests; on API quota exhaustion they pause and allow offline plots and output exports to continue. Resume incomplete requests when quota becomes available. To preserve files after runtime shutdown, download the output archive or back up `ROOT` to Drive.


In [ ]:
%pip -q install -U pymupdf tiktoken sentence-transformers faiss-cpu google-genai pydantic

In [ ]:
import os, re, json, time, math, hashlib, warnings, zipfile
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fitz
import tiktoken
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
from IPython.display import display, Markdown

def show_table(obj, rows=30):
    """Print DataFrames or structured Python objects without Pandas HTML display."""
    if isinstance(obj, pd.DataFrame):
        print(obj.head(rows).to_string(index=False))
        if len(obj) > rows:
            print(f'... showing {rows} of {len(obj)} rows')
    else:
        print(json.dumps(obj, ensure_ascii=False, indent=2, default=str))

# Save to mounted Drive by setting the optional environment variable PALAU_OUTPUT_DIR.
# Without that, results are saved under /content and must be backed up before runtime expiry.
ROOT=Path(os.environ.get('PALAU_OUTPUT_DIR', '/content/palau_imf_outputs'))
ROOT.mkdir(parents=True, exist_ok=True)
for folder in ['data','summaries','evaluation','figures','rag','prompts']:
    (ROOT/folder).mkdir(exist_ok=True)
GENERATOR_MODEL='gemini-3.8-flash'
JUDGE_MODEL='gemini-3.8-flash'  # independent judge prompt; optionally use another available model
CHUNK_SIZE=450
CHUNK_OVERLAP=50
SEED=42
np.random.seed(SEED)

def save_json(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def clean_json(s):
    s=s.strip()
    if s.startswith('```'):
        s=re.sub(r'^```(?:json)?\s*', '', s)
        s=re.sub(r'\s*```$', '', s)
    return json.loads(s)

print('Output folder:',ROOT)

### Secure key + PDF input
Set a Colab secret named `GEMINI_API_KEY` (left sidebar → key icon), or use the hidden password prompt. The PDF **must** be your assigned report; this notebook checks its title. The report file is not included in the notebook.

In [ ]:
from google.colab import files, userdata
from getpass import getpass

# Colab Secrets must contain the API key as plain text, not JSON.
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = None

if not isinstance(API_KEY, str) or not API_KEY.strip():
    print('GEMINI_API_KEY missing or not plain text in Colab Secrets.')
    API_KEY = getpass('Enter Gemini API key (hidden): ')
if not isinstance(API_KEY, str) or not API_KEY.strip():
    raise RuntimeError('A nonempty Gemini API key string is required.')
API_KEY = API_KEY.strip()
client = genai.Client(api_key=API_KEY)
print('Gemini client initialized; key not displayed.')
print('Upload the Republic of Palau IMF Country Report PDF:')
pdf_candidates = sorted(Path('/content').glob('*.pdf'))
if pdf_candidates:
    PDF_PATH = pdf_candidates[0]
    print('Reusing existing PDF (no re-upload):', PDF_PATH.name)
else:
    uploaded = files.upload()
    pdf_names = [name for name in uploaded if name.lower().endswith('.pdf')]
    if not pdf_names:
        raise ValueError('No PDF uploaded. Upload the assigned Palau report and rerun.')
    PDF_PATH = Path('/content') / pdf_names[0]
print('Input PDF:',PDF_PATH.name)

## 1 · PDF parsing, cleanup and source-linked chunking (Task 1)
The cleaning removes obvious repeated running headers, page-number artifacts and broken inline whitespace, while retaining actual content and page provenance. PDF tables are not always read perfectly as plain text; later sections incorporate **manually audited benchmark rows** with explicit page citations.

In [ ]:
doc=fitz.open(str(PDF_PATH))
raw_pages=[page.get_text('text', sort=True) for page in doc]
if len(raw_pages)<20 or not any('PALAU' in p.upper() for p in raw_pages[:8]):
    raise ValueError('This PDF does not appear to be the assigned IMF Republic of Palau report.')

def clean_page(text):
    lines=[]
    for line in text.splitlines():
        s=re.sub(r'\s+', ' ', line).strip()
        if not s: continue
        if re.fullmatch(r'(?:REPUBLIC OF PALAU|INTERNATIONAL MONETARY FUND|\d{1,2})',s,re.I):continue
        # Remove table-of-contents leaders; preserve titles and numerics
        s=re.sub(r'_{4,}', ' ', s)
        lines.append(s)
    return '\n'.join(lines)

pages=[{'page':i+1,'text':clean_page(s),'raw_chars':len(s)} for i,s in enumerate(raw_pages)]
full_text='\n\n'.join(f'[PDF PAGE {p["page"]}]\n'+p['text'] for p in pages)
(ROOT/'data'/'extracted_full_text.txt').write_text(full_text,encoding='utf-8')
page_stats=pd.DataFrame([{'page':p['page'],'characters':len(p['text'])} for p in pages])
page_stats.to_csv(ROOT/'data'/'page_statistics.csv',index=False)
print('Total PDF pages:',len(pages),'| text characters:',len(full_text))
print('Pages with very little text:',page_stats.query('characters < 100').page.tolist())
show_table(page_stats.head())

In [ ]:
enc=tiktoken.get_encoding('cl100k_base')  # approximate common BPE token counting

def make_chunks(page_records, max_tokens=450, overlap=50):
    if not 0<=overlap<max_tokens: raise ValueError('Overlap must be smaller than chunk size')
    out=[]
    for p in page_records:
        tokens=enc.encode(p['text'])
        if not tokens:continue
        step=max_tokens-overlap
        for offset in range(0,len(tokens),step):
            batch=tokens[offset:offset+max_tokens]
            if not batch:break
            out.append({'chunk_id':len(out),'page':p['page'],
                        'token_start':offset,'n_tokens':len(batch), 'text':enc.decode(batch)})
            if offset+max_tokens>=len(tokens):break
    return out
chunks=make_chunks(pages,CHUNK_SIZE,CHUNK_OVERLAP)
assert chunks and all(c['n_tokens']<=CHUNK_SIZE for c in chunks)
save_json(ROOT/'data'/'chunks.json',chunks)
print('Chunks:',len(chunks),'| mean tokens:',round(np.mean([c['n_tokens'] for c in chunks]),1))
print('Example provenance:',{k:chunks[0][k] for k in ['chunk_id','page','n_tokens']})

## 2 · Local FAISS vector retrieval (Task 3)
We embed each page-linked chunk using the local `sentence-transformers/all-MiniLM-L6-v2` model. FAISS uses normalized embeddings and inner-product search (equivalent to cosine similarity). Page citations from retrieval are retained in prompts and exported. The index is local; only text passages sent to Gemini leave the runtime.

In [ ]:
EMBED_MODEL='sentence-transformers/all-MiniLM-L6-v2'
embedder=SentenceTransformer(EMBED_MODEL)
texts=[c['text'] for c in chunks]
vectors=embedder.encode(texts,batch_size=32,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True)
vectors=np.asarray(vectors,dtype='float32')
index=faiss.IndexFlatIP(vectors.shape[1]); index.add(vectors)
faiss.write_index(index,str(ROOT/'rag'/'palau.faiss'))

def retrieve(query,k=5):
    q=np.asarray(embedder.encode([query],normalize_embeddings=True),dtype='float32')
    distances,ids=index.search(q,min(k,index.ntotal))
    return [{'score':float(score),**chunks[int(i)]} for score,i in zip(distances[0],ids[0]) if i>=0]

def context_for(query,k=5,char_limit=13500):
    hits=retrieve(query,k)
    blocks=[]; chosen=[]; total=0
    for h in hits:
        block=f'[SOURCE PDF PAGE {h["page"]}, CHUNK {h["chunk_id"]}]\n{h["text"]}'
        if total+len(block)>char_limit: continue  # never cut a citation or chunk midstream
        blocks.append(block); chosen.append(h); total+=len(block)+2
    return '\n\n'.join(blocks),chosen

query_examples=[
    'What were the main risks and recommendations about Palau financial supervision and bank capital?',
    'What is the FY2026 real GDP growth projection and FY2025 inflation?',
    'What are Palau pension and public debt reform priorities?',
    'What are the risks from the tokenized dollar and payment system?'
]
retrieval_demo=[]
for q in query_examples:
    hits=retrieve(q,5)
    retrieval_demo.append({'query':q,'retrieved_pages':[h['page'] for h in hits],
                           'similarities':[round(h['score'],3) for h in hits]})
save_json(ROOT/'rag'/'retrieval_examples.json',retrieval_demo)
show_table(pd.DataFrame(retrieval_demo))

## 3 · Grounded LLM utilities and prompts
The generator must abstain on missing fields and cite PDF page numbers; the judge uses a **separate instruction set**, independently checks the source excerpts, and outputs numeric ratings and errors. A second judge prompt is permitted by the brief; it is not an independent human factual audit. The Gemini API may have rate limits.
**Quota awareness:** 429 errors explicitly stating a *per-day* limit are NOT transient: the notebook skips that model for the remainder of the runtime. Cached prompt responses and section checkpoints reduce repeated calls. The notebook will fail clearly if all available models are exhausted; it will never invent summaries to fill gaps. Note that restarting a runtime does not reset Google quota.


In [ ]:
# Persistent response cache: identical successful prompts do not consume quota again.
# A new Colab runtime clears /content; copy this folder to Drive if you want persistence.
CACHE_DIR=ROOT/'llm_cache'
CACHE_DIR.mkdir(parents=True,exist_ok=True)
CALL_LOG=[]
BLOCKED_MODELS=set()  # Models with exhausted DAILY quota or unsupported names.
CAPACITY_MODELS=set()  # Models temporarily returning 503 during this runtime.
# Set these only to model IDs available to YOUR key. They are not guaranteed to exist.
# Do not guess whether fallback IDs exist: list models available to the key at runtime.
# Model listing does not itself generate content. If listing fails, retain the supplied primary.
try:
    available_model_ids = {m.name.replace('models/', '') for m in client.models.list()
        if 'generateContent' in (getattr(m, 'supported_actions', None) or [])}
except Exception as exc:
    available_model_ids = set()
    print('Could not list model availability; API will validate IDs:', str(exc)[:150])
preferred_fallbacks=['gemini-3.6-flash','gemini-3.5-flash-lite']
FALLBACK_MODELS=[m for m in preferred_fallbacks if not available_model_ids or m in available_model_ids]
print('Fallback model candidates:',FALLBACK_MODELS)


def call_llm(prompt,system_instruction,model=None,json_mode=False,max_tokens=1800):
    if model is None:
        model=GENERATOR_MODEL
    candidates=list(dict.fromkeys([model]+FALLBACK_MODELS))
    payload={'prompt':prompt,'system':system_instruction,'json_mode':json_mode,
             'max_tokens':max_tokens,'requested_model':model}
    key=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
    cache_file=CACHE_DIR/(key+'.json')
    if cache_file.exists():
        cached=json.loads(cache_file.read_text(encoding='utf-8'))
        print('Using cached Gemini response:',cached['model'])
        return cached['output']
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        max_output_tokens=max_tokens,
        response_mime_type='application/json' if json_mode else 'text/plain')
    last_error=None
    for candidate in candidates:
        if candidate in BLOCKED_MODELS or candidate in CAPACITY_MODELS:
            continue
        if available_model_ids and candidate not in available_model_ids:
            print('Model not listed for this API key:',candidate)
            BLOCKED_MODELS.add(candidate)
            continue
        for attempt in range(3):
            started=time.perf_counter()
            try:
                result=client.models.generate_content(
                    model=candidate, contents=prompt, config=config)
                output=(result.text or '').strip()
                if not output:
                    raise ValueError('Empty response; inspect output-token limit or finish reason.')
                record={'requested_model':model,'model':candidate,
                        'seconds':round(time.perf_counter()-started,2),
                        'prompt_chars':len(prompt),'output_chars':len(output),
                        'json_mode':json_mode}
                CALL_LOG.append(record)
                cache_file.write_text(json.dumps({'output':output,'model':candidate,
                    'record':record},ensure_ascii=False,indent=2),encoding='utf-8')
                if candidate!=model:print('Fallback used:',candidate)
                return output
            except Exception as e:
                last_error=e
                status=getattr(e,'code',None) or getattr(e,'status_code',None)
                msg=str(e).lower()
                if status==404 or '404 not_found' in msg or 'no longer available' in msg:
                    BLOCKED_MODELS.add(candidate)
                    print(candidate,'is unavailable; skipping for remainder of session.')
                    break
                if status==429 or '429 resource_exhausted' in msg:
                    if any(x in msg for x in ['per day','requests per day','daily','perprojectpermodel-freetier']) and 'per minute' not in msg:
                        BLOCKED_MODELS.add(candidate)
                        print(candidate,'has exhausted its daily quota; will not retry this session.')
                        break
                    if attempt<2:
                        delay=10*(2**attempt)
                        print(candidate,'temporarily rate limited; sleeping',delay,'seconds')
                        time.sleep(delay)
                        continue
                    BLOCKED_MODELS.add(candidate)
                    break
                if status==503 or '503 unavailable' in msg:
                    if attempt<2:
                        delay=5*(2**attempt)
                        print(candidate,'temporarily busy; sleeping',delay,'seconds')
                        time.sleep(delay)
                        continue
                    CAPACITY_MODELS.add(candidate)
                    print(candidate,'is busy; skipping it for this Colab session.')
                    break
                raise RuntimeError(f'Gemini request failed with {candidate}: {e}') from e
    raise RuntimeError(
        'No usable Gemini models remain for this session. Daily quotas do not reset with '
        'a Colab restart or a different notebook. Check your project limits at '
        'https://ai.dev/rate-limit and resume after reset, or configure a key/project '
        'with sufficient quota. Successful calls saved in '+str(CACHE_DIR)+
        '. Last error: '+str(last_error)[:350])

GENERATE_SYSTEM=('You are a careful macroeconomic research assistant. Use ONLY supplied IMF source '
                 'passages. Do not invent a statistic or source. State pages in each finding; '
                 'separate estimates/projections from historical observations, and keep time periods exact.')
JUDGE_SYSTEM=('You are an adversarial factual auditor, not a rewriting assistant. Evaluate ONLY '
              'against supplied IMF passages, checking each number, period, population and unit. '
              'Abstain from judging evidence not supplied. Return valid JSON.')

SUMMARY_PROMPT=('Summarize the supplied source excerpts with 5-7 concise, factual executive bullets '
                'about economic growth, financial stability, fiscal conditions and downside risks. '
                'Put source PDF page citations in each bullet. Do not claim the excerpts are the entire document.\n\n')
AUDIT_PROMPT=('Audit the candidate output against the cited evidence. Return JSON with exactly: '
              '{"consistency": integer 0-5,"completeness": integer 0-5,'
              '"factual_alignment": integer 0-5,"unsupported_claims": array of strings,'
              '"review_notes": array of strings}. Do not assume missing source context proves an error.\n\n')
save_json(ROOT/'prompts'/'prompt_templates.json',{'generator_system':GENERATE_SYSTEM,
 'judge_system':JUDGE_SYSTEM,'summary':SUMMARY_PROMPT,'audit':AUDIT_PROMPT})

## 4 · Executive summary and per-section summaries (Task 1)
To represent the **entire** report rather than just the first pages, use a section-aware map-reduce strategy: extract short source windows from each main section, summarize those, then retrieve across report topics for final executive synthesis. The TOC sections are taken from the actual Palau staff report; annexes are also summarised as a collection. Every summary is an AI-produced claim requiring source checking.

Section boundaries below are **PDF page ranges**, inclusive, based on the document's contents; inspect if a different edition is uploaded.

In [ ]:
SECTIONS={
 'Context':(9,9),
 'Recent developments':(9,11),
 'Outlook and risks':(11,12),
 'Policy discussions — fiscal sustainability':(12,16),
 'Policy discussions — financial markets':(16,19),
 'Policy discussions — sustainable growth':(19,21),
 'Capacity development and data issues':(21,22),
 'Staff appraisal':(22,23),
 'Statistical tables and financial soundness':(24,32),
 'Annexes — tax, fiscal, labor, external, risks, debt, data':(33,71)
}

def section_evidence(start, end, max_chars=12500):
    """Represent each page without silently limiting long sections to its first lines.

    Small sections: entire pages within budget. Long annexes: deterministic
    beginning/middle/end snippets plus retrieval of relevant chunks.
    Each snippet carries its true PDF page number. No invented citations.
    """
    selected=[p for p in pages if start <= p['page'] <= end and p['text'].strip()]
    if not selected: return ''
    full_blocks=[f'[PDF PAGE {p["page"]}]\n{p["text"]}' for p in selected]
    if sum(map(len,full_blocks)) <= max_chars:
        return '\n\n'.join(full_blocks)
    # Split budget among all pages and diversify start/end coverage.
    budget=max(180, max_chars // len(selected)-38)
    blocks=[]
    for p in selected:
        content=p['text']
        if len(content)<=budget:
            sample=content
        else:
            first=budget//2; last=budget-first
            sample=content[:first]+'\n[... middle of page omitted ...]\n'+content[-last:]
        blocks.append(f'[PDF PAGE {p["page"]}]\n{sample}')
    return '\n\n'.join(blocks)[:max_chars]

# Annexes contain many independent topics; provide a page-level coverage log to
# distinguish a broad sketch from a full reading of every annex.
section_coverage={name:{'pdf_start':a,'pdf_end':b,
    'total_pages':b-a+1,'evidence_chars':len(section_evidence(a,b))}
    for name,(a,b) in SECTIONS.items()}
save_json(ROOT/'evaluation'/'section_coverage.json',section_coverage)

checkpoint_file=ROOT/'summaries'/'section_summaries.json'
section_summaries=json.loads(checkpoint_file.read_text(encoding='utf-8')) if checkpoint_file.exists() else []
# Earlier runs sometimes produced 7- or 8-word answers. Treat them as incomplete.
def valid_section_summary(item):
    return (isinstance(item.get('summary'),str) and
        25 <= len(item['summary'].split()) < 100 and
        bool(re.search(r'\b(?:page|p\.)\s*\d+',item['summary'],re.I)))
section_summaries=[x for x in section_summaries if valid_section_summary(x)]
done_sections={x['section'] for x in section_summaries}
for name,(a,b) in SECTIONS.items():
    if name in done_sections:
        print('Already completed:',name)
        continue
    source=section_evidence(a,b)
    request=(f'Section name: {name}. Summarise in FEWER THAN 100 WORDS. '
             'Highlight financial/economic findings and qualification. Cite PDF page numbers. '
             'Discuss only supplied pages; if tables are hard to parse, say so.\n\n'+source)
    response=call_llm(request,GENERATE_SYSTEM,max_tokens=430)
    if len(response.split()) < 25 or len(response.split()) >= 100:
        print(f'REVIEW REQUIRED: {name} has {len(response.split())} words; not checkpointing as complete.')
        (ROOT/'summaries'/('needs_review_'+str(a)+'.txt')).write_text(response,encoding='utf-8')
        continue
    section_summaries.append({'section':name,'pdf_page_start':a,'pdf_page_end':b,
                              'summary':response,'word_count':len(response.split())})
    save_json(checkpoint_file,section_summaries)  # Save AFTER EACH success
    print(f'{name}: {len(response.split())} words')
    if len(response.split())>=100:print('  REVIEW: over word limit; shorten manually or rerun.')
save_json(ROOT/'summaries'/'section_summaries.json',section_summaries)
show_table(pd.DataFrame(section_summaries)[['section','summary','word_count']])

In [ ]:
topics=[
 'tourism and construction recovery growth outlook FY2025 FY2026',
 'government debt fiscal reform compact trust fund pension risks',
 'banks NPL capital supervision national development bank payment systems',
 'inflation commodity prices and uncertainty',
 'climate change risks migration labor shortages structural diversification',
 'debt sustainability analysis and external sector challenges'
]
source_blocks=[]
for topic in topics:
    context,_=context_for(topic,k=3,char_limit=4000)
    source_blocks.append('TOPIC: '+topic+'\n'+context)
executive_source='\n\n'.join(source_blocks)
executive=call_llm(SUMMARY_PROMPT+executive_source,GENERATE_SYSTEM,max_tokens=1200)
(ROOT/'summaries'/'executive_summary.md').write_text(executive,encoding='utf-8')
display(Markdown('### Source-grounded executive summary\n'+executive))

## 5 · Seven financial-stability themes and strengths/challenges (Task 2)
**Operational definition:** case-insensitive phrase occurrences in the cleaned document, not probability of risk or unique paragraphs. Related patterns within a phrase family are counted once per match and thematic matches may overlap. Results should be interpreted as **mention density**, not economic importance.

In [ ]:
THEMES={
 'Inflation & Prices':[r'\binflation\b',r'\bconsumer pric(?:e|es)\b',r'\bcommodity pric(?:e|es)\b',r'\bcost of living\b'],
 'Interest Rates & Credit':[r'\binterest rates?\b',r'\bcredit\b',r'\blending\b',r'\bloan(?:s)?\b'],
 'Housing & Real Estate':[r'\bhousing\b',r'\breal estate\b',r'\bproperty\b',r'\bhotel construction\b'],
 'Banking Capital & Liquidity':[r'\bbank(?:s|ing)?\b',r'\bcapital (?:buffer|adequacy|ratio)\b',r'\bliquidity\b',r'\bnonperforming loans?\b',r'\bnon-performing loans?\b'],
 'Sovereign Debt':[r'\bdebt\b',r'\bfiscal sustainability\b',r'\bborrowing\b',r'\bdebt anchor\b'],
 'Labor Market Dynamics':[r'\blabou?r\b',r'\bemployment\b',r'\bwages?\b',r'\bunemployment\b',r'\bemigration\b'],
 'Systemic Financial Risk':[r'\bfinancial stability\b',r'\bsystemic\b',r'\bfinancial risk\b',r'\bpayment system\b',r'\bfinancial integrity\b']
}
count_text='\n'.join(p['text'] for p in pages)
records=[]
for theme,patterns in THEMES.items():
    matches=sum(len(re.findall(pattern,count_text,re.I)) for pattern in patterns)
    records.append({'theme':theme,'occurrences':matches,'per_10000_words':round(matches/max(len(count_text.split()),1)*10000,2)})
themes_df=pd.DataFrame(records).sort_values('occurrences',ascending=False)
themes_df['share_of_theme_matches_pct']=(100*themes_df.occurrences/themes_df.occurrences.sum()).round(2)
themes_df.to_csv(ROOT/'data'/'thematic_counts.csv',index=False)
save_json(ROOT/'data'/'thematic_counts.json',themes_df.to_dict('records'))
show_table(themes_df)
print('Interpretation: sum of category matches may exceed distinct mentions.')

In [ ]:
# Qualitative insights: loads valid prior results before spending another API call.
qual_path=ROOT/'data'/'qualitative_insights.json'
qual=None
if qual_path.exists():
    try: qual=json.loads(qual_path.read_text(encoding='utf-8'))
    except (OSError,json.JSONDecodeError): qual=None

def valid_qual(q):
    return isinstance(q,dict) and all(isinstance(q.get(k),list) and
        1<=len(q[k])<=7 and all(isinstance(it,dict) and
        isinstance(it.get('item'),str) and len(it['item'])>12 and
        isinstance(it.get('page'),int) and 1<=it['page']<=len(pages)
        for it in q[k]) for k in ['strengths','vulnerabilities'])

if not valid_qual(qual):
    risks_context='\n\n'.join(context_for(q,k=4,char_limit=6600)[0] for q in [
      'Palau tourism construction compact grants tax reform recovery strengths',
      'Palau pension climate shocks labor migration bank supervision tokenized dollar vulnerabilities'])
    prompt=('Extract 5-7 strengths and 5-7 vulnerabilities, or fewer if not supported. '
      'Return JSON exactly {"strengths":[{"item":"...","page":2}],'
      '"vulnerabilities":[{"item":"...","page":2}]}. '
      'Each page must occur in its cited source passage. Keep findings distinct; '
      'do not claim plans are achieved outcomes. Source evidence:\n'+risks_context)
    try:
        candidate=clean_json(call_llm(prompt,GENERATE_SYSTEM,json_mode=True,max_tokens=1700))
        if not valid_qual(candidate):
            raise ValueError('Qualitative JSON failed structural/page validation; inspect the raw response.')
        qual=candidate
        save_json(qual_path,qual)
    except RuntimeError as exc:
        if any(x in str(exc).lower() for x in ['quota','rate limit','resource_exhausted','no usable gemini']):
            print('Qualitative extraction pending: API quota not available. No fabricated output.')
        else: raise
if valid_qual(qual):
    show_table(qual)
else:
    print('Qualitative insights unavailable; this deliverable is incomplete until a valid saved extraction exists.')


## 6 · Strict JSON financial indicators + documented fallbacks (Task 2)
**Target periods for an unambiguous comparison:** GDP growth FY2026 **forecast**; headline inflation FY2025 **estimate**; latest clearly tabulated **commercial-bank** NPL and Tier 1-to-risk-weighted-assets are 2023. Unemployment and domestic policy rate may be unavailable/not applicable. A `-1.0` value flags unavailable or not applicable and is **never plotted as an actual percentage**.

**Important:** Table 5 (PDF page 32) explicitly **excludes NDBP**. The **0.8% NDBP NPL** shown in discussion for FY2024 has a different institution and period. This notebook records both descriptions separately.

Raw Gemini values are saved separately from explicitly justified human-rule normalization. Accuracy must be evaluated on raw values, not silently corrected values.


In [ ]:
INDICATOR_KEYS=[
 'country_name','gdp_growth_forecast_pct','headline_inflation_rate_pct',
 'central_bank_policy_rate_pct','non_performing_loan_ratio',
 'unemployment_rate_pct','tier_1_capital_ratio'
]
class Indicators(BaseModel):
    country_name:str
    gdp_growth_forecast_pct:float
    headline_inflation_rate_pct:float
    central_bank_policy_rate_pct:float
    non_performing_loan_ratio:float
    unemployment_rate_pct:float
    tier_1_capital_ratio:float

# Hand-checked benchmark transcribed from the uploaded IMF report, NOT model predictions.
# PDF p5 Table 1; PDF p32 Table 5 (commercial banks only, excl. NDBP).
AUDITED_REFERENCE={
 'country_name':'Republic of Palau',
 'gdp_growth_forecast_pct':4.1,
 'headline_inflation_rate_pct':0.2,
 'central_bank_policy_rate_pct':-1.0,
 'non_performing_loan_ratio':0.61,
 'unemployment_rate_pct':-1.0,
 'tier_1_capital_ratio':0.08
}
REFERENCE_DETAILS={
 'gdp_growth_forecast_pct':{'period':'FY2026','scope':'real GDP growth forecast','pdf_page':5,'status':'projected'},
 'headline_inflation_rate_pct':{'period':'FY2025','scope':'consumer price inflation annual average','pdf_page':5,'status':'estimated'},
 'central_bank_policy_rate_pct':{'period':None,'scope':'Palau domestic policy rate','pdf_page':None,'status':'not applicable','reason':'Palau uses USD and has no independently set domestic central-bank policy rate'},
 'non_performing_loan_ratio':{'period':'2023','scope':'commercial banks excluding NDBP','pdf_page':32,'status':'reported'},
 'unemployment_rate_pct':{'period':None,'scope':'national unemployment rate','pdf_page':None,'status':'not located in assigned report; verify manually'},
 'tier_1_capital_ratio':{'period':'2023','scope':'Tier 1 capital / risk-weighted assets, commercial banks excluding NDBP','pdf_page':32,'status':'reported'}
}

# The numerical-table context is preserved verbatim from PDF extraction.
indicator_evidence='\n\n'.join(f'[PDF PAGE {p}]\n{pages[p-1]["text"]}' for p in [5,16,32])
indicator_prompt="""Extract one JSON object with exactly these fields:
 country_name (string), gdp_growth_forecast_pct (float), headline_inflation_rate_pct (float),
 central_bank_policy_rate_pct (float), non_performing_loan_ratio (float),
 unemployment_rate_pct (float), tier_1_capital_ratio (float).
Selection rules: GDP FY2026 forecast, headline inflation FY2025 annual average,
 commercial bank NPL ratio for 2023, commercial bank Tier 1 to RISK-WEIGHTED ASSETS for 2023.
 Table 5 EXCLUDES NDBP, so never use its separate NPL number. Never mistake Tier 1 to
 total assets for Tier 1 to risk-weighted assets. If unavailable or inapplicable put -1.0.
 Do not use any external knowledge or invent a policy rate or unemployment statistic.
 Return ONLY valid JSON, without explanation. SOURCE:
"""
raw_indicators=call_llm(indicator_prompt+indicator_evidence,GENERATE_SYSTEM,json_mode=True,max_tokens=600)
parsed=clean_json(raw_indicators)
raw_indicators_validated=Indicators.model_validate(parsed).model_dump()
assert set(raw_indicators_validated)==set(INDICATOR_KEYS)
save_json(ROOT/'data'/'core_indicators_raw_llm.json',raw_indicators_validated)
# Explicit human-rule normalization is distinct from the raw model prediction.
indicators=dict(raw_indicators_validated)
corrections=[]
for key in ('central_bank_policy_rate_pct','unemployment_rate_pct'):
    if indicators[key] != -1.0:
        corrections.append({'field':key,'model_value':indicators[key],
          'verified_value':-1.0,'reason':REFERENCE_DETAILS[key]['status']})
    indicators[key]=-1.0
save_json(ROOT/'data'/'post_extraction_corrections.json',corrections)
save_json(ROOT/'data'/'core_indicators.json',indicators)
save_json(ROOT/'data'/'indicator_provenance_and_fallbacks.json',REFERENCE_DETAILS)
save_json(ROOT/'data'/'audited_reference_values.json',AUDITED_REFERENCE)
print(json.dumps(indicators,indent=2))

### Automated benchmark comparison (not a substitute for human checking)
Compare extracted values to hand-audited IMF reference values, report exact matches and missingness, and inspect anomalous values manually. The benchmark explicitly checks **period and population choices**. These source references are visible so they can be verified by the marker.

In [ ]:
# Accurate evaluation: available metrics, fallbacks and human corrections are separated.
benchmark_rows=[]
for key in INDICATOR_KEYS:
    observed=raw_indicators_validated[key]  # NOT the manually normalized version
    expected=AUDITED_REFERENCE[key]
    match=(str(observed).casefold()==str(expected).casefold()) if isinstance(expected,str) else \
        math.isclose(float(observed),float(expected),abs_tol=0.015)
    benchmark_rows.append({'indicator':key,'raw_llm_value':observed,
        'audited_reference':expected,'match':match,
        'is_missing_or_inapplicable':key in ('central_bank_policy_rate_pct','unemployment_rate_pct'),
        'period':REFERENCE_DETAILS.get(key,{}).get('period'),
        'pdf_page':REFERENCE_DETAILS.get(key,{}).get('pdf_page')})
benchmark_df=pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(ROOT/'evaluation'/'numeric_accuracy.csv',index=False)
observed=benchmark_df[~benchmark_df.is_missing_or_inapplicable]
missing=benchmark_df[benchmark_df.is_missing_or_inapplicable]
print(f'Raw extraction agreement on observed fields: {observed.match.sum()}/{len(observed)}')
print(f'Correct fallback treatment: {missing.match.sum()}/{len(missing)}')
print(f'Post-extraction human-rule corrections: {len(corrections)}')
show_table(benchmark_df)
if not observed.match.all():
    print('REVIEW REQUIRED: extracted values disagree with manually checked source tables.')


## 7 · Secondary LLM-as-a-Judge framework (Tasks 1–3)
The second **role/prompt** audits summaries, indicators, and grounded query answers. Using the same model family with a separate adversarial prompt satisfies the assignment's alternative of an automated judge loop, but cannot rule out shared model errors. Record both the scores and any specific criticisms.

In [ ]:
# Incremental judge checkpoint: do NOT discard completed work when quota runs out.
def judge(candidate, source, task, max_tokens=850):
    prompt=(AUDIT_PROMPT+'TASK: '+task+'\nCANDIDATE:\n'+str(candidate)[:9000]
            +'\nSOURCE EVIDENCE:\n'+source[:14000])
    obj=clean_json(call_llm(prompt,JUDGE_SYSTEM,model=JUDGE_MODEL,json_mode=True,max_tokens=max_tokens))
    for k in ['consistency','completeness','factual_alignment']:
        obj[k]=int(obj.get(k,0))
        if not 0<=obj[k]<=5: raise ValueError(f'Invalid judge score: {k}')
    obj['task']=task
    return obj

judge_checkpoint=ROOT/'evaluation'/'judge_scores.json'
judgements=json.loads(judge_checkpoint.read_text(encoding='utf-8')) if judge_checkpoint.exists() else []
completed_tasks={r['task'] for r in judgements}
judge_jobs=[('Executive summary grounding',executive,executive_source),
            ('Core indicator schema and factual alignment',json.dumps(indicators),indicator_evidence)]
for s in section_summaries:
    a,b=SECTIONS[s['section']]
    judge_jobs.append(('Section: '+s['section'],s['summary'],section_evidence(a,b,7500)))
quota_paused=False
for task,candidate,evidence in judge_jobs:
    if task in completed_tasks:
        print('Already judged:',task)
        continue
    try:
        result=judge(candidate,evidence,task)
    except RuntimeError as exc:
        if any(term in str(exc).lower() for term in ['quota','rate limit','no usable gemini models','resource_exhausted']):
            print('API quota exhausted. Pausing judge evaluation; previous scores are saved.')
            quota_paused=True
            break
        raise
    judgements.append(result)
    completed_tasks.add(task)
    save_json(judge_checkpoint,judgements)  # Save immediately after each response
    print('Saved judge result:',task)
judge_df=pd.DataFrame(judgements)
if len(judge_df):
    show_table(judge_df[[c for c in ['task','consistency','completeness','factual_alignment','unsupported_claims'] if c in judge_df]])
else:
    print('No LLM judge scores yet. No judge scores will be invented.')
print(f'Judge evaluations: {len(judgements)}/{len(judge_jobs)} complete; missing:',len(judge_jobs)-len(judgements))


### RAG question-answering and retrieval evaluation
The **baseline** is a grounded question with one retrieval query; the **multi-query extension** expands each question into three related searches. Both are audited with the same judge. The source pages are captured so you can verify whether the evidence was actually relevant.

In [ ]:
# Save each answer+judge result separately so a quota failure cannot erase earlier results.
def rag_answer(question,multiquery=False):
    queries=[question]
    if multiquery:
        queries += [question.replace('What','Explain').replace('?', ' and relevant supporting evidence?'),
                    question+' IMF staff recommendation policy risks']
    hits=[]; seen=set()
    for q in queries:
        for hit in retrieve(q,4):
            if hit['chunk_id'] not in seen:
                hits.append(hit); seen.add(hit['chunk_id'])
    hits=hits[:9 if multiquery else 4]
    context='\n\n'.join(f'[PDF PAGE {h["page"]}, CHUNK {h["chunk_id"]}] {h["text"]}' for h in hits)
    strategy='multi-query' if multiquery else 'single-query'
    # Cache an answer before judging it; quota loss on judge no longer loses the answer.
    answer_path=ROOT/'rag'/('answer_'+hashlib.sha256((question+strategy).encode()).hexdigest()[:16]+'.json')
    saved=json.loads(answer_path.read_text(encoding='utf-8')) if answer_path.exists() else None
    if saved is None:
        reply=call_llm('Answer in 120-180 words with PDF page citations. If evidence '
          'is insufficient, abstain. QUESTION: '+question+'\nSOURCE:\n'+context,
          GENERATE_SYSTEM,max_tokens=450)
        saved={'question':question,'strategy':strategy,'answer':reply,
            'retrieved_pages':list(dict.fromkeys(h['page'] for h in hits)),
            'retrieved_chunk_ids':[h['chunk_id'] for h in hits]}
        save_json(answer_path,saved)
    if 'judge' not in saved:
        saved['judge']=judge(saved['answer'],context,'RAG '+strategy+': '+question,600)
        save_json(answer_path,saved)
    return saved

rag_questions=[
 'What reforms does the IMF recommend to improve bank supervision and manage NDBP risks?',
 'What are the main threats to Palau from the tokenized dollar and domestic payment arrangements?',
 'How does the IMF expect Palau to reduce debt and address pension liabilities?'
]
rag_checkpoint=ROOT/'rag'/'rag_answers_and_evaluation.json'
rag_runs=json.loads(rag_checkpoint.read_text(encoding='utf-8')) if rag_checkpoint.exists() else []
finished={(r['question'],r['strategy']) for r in rag_runs}
if quota_paused:
    print('Quota exhausted during judge evaluation: skipping new RAG API calls for now.')
else:
    for question in rag_questions:
        for mode in [False,True]:
            strategy='multi-query' if mode else 'single-query'
            if (question,strategy) in finished: continue
            try:
                result=rag_answer(question,mode)
            except RuntimeError as exc:
                if any(term in str(exc).lower() for term in ['quota','rate limit','no usable gemini models','resource_exhausted']):
                    quota_paused=True
                    print('API limit reached in RAG; prior responses saved. Resume after quota reset.')
                    break
                raise
            rag_runs.append(result)
            save_json(rag_checkpoint,rag_runs)
        if quota_paused: break
if rag_runs:
    rag_eval=pd.DataFrame([{'question':r['question'],'strategy':r['strategy'],
      'factual_alignment':r['judge']['factual_alignment'],'completeness':r['judge']['completeness'],
      'consistency':r['judge']['consistency'],'n_pages':len(r['retrieved_pages'])} for r in rag_runs])
    show_table(rag_eval)
    show_table(rag_eval.groupby('strategy')[['factual_alignment','completeness','consistency']].mean())
else:
    print('No RAG generation/evaluation yet; the offline FAISS retrieval index is still available.')


## 8 · Source-checked historical and forecast time series (Task 2)
The numerical rows below are **manual transcriptions** of *Table 1: Selected Economic Indicators* (PDF page 5); they are clearly labelled as an audited reference, not as values secretly generated by the LLM. The table uses **FY2022/23 through FY2029/30**, with later columns projected. The indicator year classification is based on the table's `Est.`/`Proj.` headings; verify the source before final submission.

In [ ]:
YEARS=['FY2023','FY2024','FY2025','FY2026','FY2027','FY2028','FY2029','FY2030']
status=['historical','historical','estimated','projected','projected','projected','projected','projected']
macro_wide=pd.DataFrame({
 'year':YEARS,'status':status,
 'GDP growth (%)':[1.2,12.0,6.7,4.1,2.7,2.6,2.2,2.2],
 'Inflation (%)':[12.4,3.6,0.2,2.8,2.7,2.5,2.3,2.2],
 'Public debt (% GDP)':[77.9,63.3,56.1,50.2,46.0,42.7,40.0,38.0]
})
macro_long=pd.melt(macro_wide,id_vars=['year','status'],
                   value_vars=['GDP growth (%)','Inflation (%)','Public debt (% GDP)'],
                   var_name='indicator',value_name='value')
macro_wide.to_csv(ROOT/'data'/'macro_timeseries_wide.csv',index=False)
macro_long.to_csv(ROOT/'data'/'macro_timeseries_long.csv',index=False)
# Table 5 (PDF p32) — commercial banks excluding NDBP, calendar years 2018–2023.
banks=pd.DataFrame({'year':[2018,2019,2020,2021,2022,2023],
 'NPL to total loans (%)':[0.60,0.66,1.56,0.31,0.60,0.61],
 'Tier 1 / risk-weighted assets (%)':[2.27,2.22,1.74,0.39,0.15,0.08],
 'Tier 1 / assets (%)':[1.27,1.39,1.10,1.08,1.15,1.25]})
banks.to_csv(ROOT/'data'/'banks_table5_timeseries.csv',index=False)
show_table(macro_long.head(9));show_table(banks)
assert macro_wide.loc[macro_wide.year=='FY2026','GDP growth (%)'].iloc[0]==4.1
assert banks.iloc[-1]['NPL to total loans (%)']==0.61

## 9 · Four-plus publication-ready dashboard plots (Task 4)
All figures have labels, title, legends where appropriate, grids, and are saved at **320 dpi**. We use distinct themes, time series, LLM judge diagnostics, and commercial-bank soundness trends. All charts use real report data or actual scores from notebook execution—**no fabricated model comparison scores**. The fourth plot does not mix incompatible NDBP and commercial-bank populations.

In [ ]:
plt.rcParams.update({'figure.figsize':(10,5.6),'font.size':11,'axes.titlesize':14,
                     'axes.labelsize':11,'savefig.bbox':'tight'})
def finish_plot(filename):
    plt.grid(alpha=.24)
    plt.tight_layout()
    plt.savefig(ROOT/'figures'/filename,dpi=320,facecolor='white')
    plt.show()

# Plot 1: seven thematic occurrences
fig,ax=plt.subplots(figsize=(11,5.5))
ordered=themes_df.sort_values('occurrences')
ax.barh(ordered.theme,ordered.occurrences,color='#277da1',label='Regex phrase matches')
ax.set(xlabel='Number of phrase occurrences in cleaned PDF',title='Palau IMF report: seven-theme mention density')
ax.legend(loc='lower right');finish_plot('01_theme_distribution.png')

# Plot 2: real GDP and inflation (same units); estimated/projected division marked
fig,ax=plt.subplots(figsize=(10.5,5.4))
for col,colr in [('GDP growth (%)','#277da1'),('Inflation (%)','#e76f51')]:
    ax.plot(macro_wide.year,macro_wide[col],marker='o',linewidth=2.1,label=col,color=colr)
ax.axvline(x=2.5,color='grey',linestyle='--',label='Forecast begins FY2026')
ax.set(ylabel='Percent',xlabel='Fiscal year ending September 30',title='IMF real growth and inflation: history, estimate and forecast')
ax.tick_params(axis='x',rotation=35);ax.legend();finish_plot('02_gdp_inflation_timeseries.png')

# Plot 3: reproducible numerical extraction diagnostic (always available after Task 2)
fig,ax=plt.subplots(figsize=(10.5,5.5))
valid_metrics=benchmark_df[(~benchmark_df.is_missing_or_inapplicable) & (benchmark_df.indicator != 'country_name')].copy()
ax.barh(valid_metrics['indicator'],valid_metrics['raw_llm_value'].astype(float),
    alpha=.8,label='Raw LLM extracted')
ax.scatter(valid_metrics['audited_reference'].astype(float),
    valid_metrics['indicator'],marker='D',s=50,color='#b53d3d',label='Table-verified reference',zorder=4)
ax.set(xlabel='Reported percentage (%)',title='Numerical extraction versus audited source')
ax.legend();finish_plot('03_numeric_reference_comparison.png')

# Plot 3: real LLM judge scores only; do not fabricate scores if quota blocked evaluation.
fig,ax=plt.subplots(figsize=(10.5,5.5))
score_cols=['consistency','completeness','factual_alignment']
if len(judge_df) and all(col in judge_df for col in score_cols):
    score_summary=judge_df[score_cols].mean()
    ax.bar(score_summary.index,score_summary.values,color=['#457b9d','#2a9d8f','#e9c46a'],label='Observed judge scores')
    ax.set(ylim=(0,5.4),ylabel='Mean LLM judge score (0–5)',
           title=f'Secondary LLM audit ({len(judge_df)}/{len(judge_jobs)} tasks completed)')
    for j,v in enumerate(score_summary.values): ax.text(j,v+.08,f'{v:.2f}',ha='center')
    ax.legend()
    finish_plot('06_llm_judge_scores.png')
else:
    plt.close(fig)
    print('Judge chart skipped: no real scores collected. Complete evaluation after quota reset.')

# Plot 4: commercial banking data (consistent reporting universe)
fig,ax=plt.subplots(figsize=(10.5,5.5))
ax.plot(banks.year,banks['NPL to total loans (%)'],marker='o',linewidth=2,label='NPL / total loans (%)',color='#e76f51')
ax.plot(banks.year,banks['Tier 1 / risk-weighted assets (%)'],marker='s',linewidth=2,
        label='Tier 1 / risk-weighted assets (%)',color='#277da1')
ax.set(xlabel='Calendar year',ylabel='Percent',title='Palau commercial banks (excl. NDBP): asset quality and capital')
ax.legend();finish_plot('04_bank_npl_tier1_timeseries.png')

# Plot 5 (bonus): public debt sustainability trend
fig,ax=plt.subplots(figsize=(10.5,5.4))
ax.plot(macro_wide.year,macro_wide['Public debt (% GDP)'],marker='o',color='#6a4c93',label='Government debt (% GDP)')
ax.axhline(30,color='#e76f51',linestyle='--',label='Legislated debt anchor (30% GDP)')
ax.axvline(2.5,color='grey',linestyle=':',label='Projection period starts')
ax.set(xlabel='Fiscal year',ylabel='Percent of GDP',title='Debt reduction trajectory versus Palau debt anchor')
ax.tick_params(axis='x',rotation=35);ax.legend();finish_plot('05_debt_anchor.png')
print('Saved',len(list((ROOT/'figures').glob('*.png'))),'charts at 320 dpi')

## 10 · Optional extra-credit extension: strict vs generic prompting
The brief permits comparison of a **strict system instruction** against a generic user-prompt-only approach. Here we run both configurations on **the same table evidence**, recording schema compliance, extraction latency and reference accuracy. Do not confuse two prompting configurations with two independent model weights. Model behaviour and judge ratings can vary across runs.

In [ ]:
def benchmark_configuration(label,system,user_prompt):
    start=time.perf_counter(); output=''; error=None; obj=None
    try:
        output=call_llm(user_prompt,system,json_mode=True,max_tokens=750)
        obj=Indicators.model_validate(clean_json(output)).model_dump()
        valid=True
    except Exception as ex:
        valid=False;error=str(ex)
    elapsed=time.perf_counter()-start
    if obj:
        accuracy=np.mean([math.isclose(float(obj[k]),float(AUDITED_REFERENCE[k]),abs_tol=.015)
                    for k in INDICATOR_KEYS if k!='country_name'])
    else:accuracy=0.0
    return {'setting':label,'schema_compliant':valid,'latency_seconds':round(elapsed,2),
            'numerical_reference_accuracy':round(float(accuracy),3),
            'error':error,'output':obj or output[:1000]}

benchmark_path=ROOT/'evaluation'/'prompt_benchmark.json'
benchmark_configs=json.loads(benchmark_path.read_text(encoding='utf-8')) if benchmark_path.exists() else []
if quota_paused:
    print('Skipping optional prompting benchmark while Gemini quota is unavailable.')
else:
    existing={r['setting'] for r in benchmark_configs}
    for label,system,user_prompt in [
      ('strict system + explicit rules',GENERATE_SYSTEM,indicator_prompt+indicator_evidence),
      ('generic system + loose request','You are a helpful assistant.',
       'Extract the country, GDP growth, inflation, monetary policy rate, bank NPL, unemployment, and Tier 1 ratio '
       'as a JSON object using the standard keys: '+', '.join(INDICATOR_KEYS)+'. Use -1.0 if unavailable.\n'+indicator_evidence)]:
        if label in existing: continue
        result=benchmark_configuration(label,system,user_prompt)
        benchmark_configs.append(result)
        save_json(benchmark_path,benchmark_configs)
        if result.get('error') and any(term in str(result['error']).lower() for term in ['quota','rate limit','no usable gemini models','resource_exhausted']):
            quota_paused=True
            print('Optional benchmark paused due to API quota.')
            break
if benchmark_configs:
    show_table(pd.DataFrame(benchmark_configs).drop(columns=['output']))
else:
    print('No prompting benchmark run yet; no results fabricated.')
print('Single-run benchmarks are illustrative only, not statistical proof.')


## 11 · Manual verification checklist, output archive and interpretation notes
Before submitting, independently inspect the report at **PDF pages 5, 16, 32** and cross-check all generated summaries and RAG citations. A high judge score does **not** establish factual correctness. Annotate your own methodological observations in your **human-written** PDF report.

**Suggested discussion topics:**
- Table extraction is harder than extracting paragraphs; commercial bank and NDBP data differ in scope.
- 2023 bank capital is an older series than FY2026 GDP projections; mixed periods should be disclosed.
- Missing or non-applicable policy/unemployment rates cannot be replaced with fabricated zero values.
- Keyword counts can overstate importance; retrieval similarity is not evidence of causal relevance.
- Multi-query RAG adds more retrieved evidence, but may also add irrelevant text and cost.
- LLM-as-a-Judge is informative but may share errors with the generation model.

**Report:** 5–6 pages maximum, written by you, covering methodology, optimized prompts, validation failures, dashboard inferences, citations and human oversight. Course instructions say AI-generated reports receive zero marks. Note the brief also requires accessible Colab/GitHub code; do not publish the report in a public repository before the course ends.

For the written report, explain when a model prediction was corrected, the difference between measured and projected indicators, what financial-soundness table exclusions imply, which pages were not fully visible to the judge, and why automatic judge scores do not guarantee truth.


In [ ]:
review_items={
 'source_file':PDF_PATH.name,'source_pages':len(pages),
 'runtime_utc':datetime.now(timezone.utc).isoformat(),
 'model_generator':GENERATOR_MODEL,'model_judge':JUDGE_MODEL,
 'chunk_size':CHUNK_SIZE,'chunk_overlap':CHUNK_OVERLAP,
 'numeric_values_require_manual_review':benchmark_df.loc[~benchmark_df['match'],'indicator'].tolist(),
 'manual_checks':[
  'Verify country and assigned year range in Canvas allocation table',
  'Verify selected indicators in PDF page 5 Table 1',
  'Verify 2023 NPL and Tier 1 ratios in PDF page 32 Table 5; excludes NDBP',
  'Confirm FY2024 NDBP NPL scope separately in PDF page 16',
  'Read executive and section summaries and check page citations',
  'Check the accuracy/limitations of thematic regex and RAG answers',
  'Discuss unavailable domestic policy rate and missing unemployment in own report',
  'Use your own written 5–6-page report, not an AI-written report'
 ]}
save_json(ROOT/'evaluation'/'human_review_checklist.json',review_items)
call_log_path=ROOT/'evaluation'/'api_call_log.json';save_json(call_log_path,CALL_LOG)

readme="""# Palau IMF PDF-to-Dashboard — execution notes
1. Open the accompanying .ipynb in Google Colab.
2. Run setup cells, supply a Gemini API key securely and upload the assigned Palau IMF report.
3. Run all cells and check the output/ figures and evaluation scores.
4. Inspect source tables (PDF p5 and p32) and manually verify AI-generated claims.
5. Submit an accessible Colab notebook link or public GitHub code as requested by the assignment.

Input: IMF Country Report 26/22, Republic of Palau, 2025 Article IV Consultation (Jan 2026).
Output: data/*.json, summaries/*.json, evaluation/*.json, rag/*.json, figures/*.png.
Key limitations: commercial-bank table excludes NDBP; unavailable/inapplicable metrics use -1.0;
LLM-as-a-Judge does not replace human verification; external API use may be billed.
"""
(ROOT/'README.md').write_text(readme,encoding='utf-8')
zip_path=Path('/content/palau_imf_results.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in ROOT.rglob('*'):
        if path.is_file():z.write(path,arcname=path.relative_to(ROOT))
print('Output archive:',zip_path,'size (MB)',round(zip_path.stat().st_size/1e6,2))
from google.colab import files
files.download(str(zip_path))

## 12 · Submission readiness audit (non-LLM)
This final gate makes missing deliverables **visible**. It does not claim an unexecuted model call succeeded. Run it after the notebook has finished, inspect all warnings, and tick off the human review checklist before submission. API quota failures can leave real deliverables pending even when the offline plots work.

In [ ]:
# Report what is actually present; do not silently treat missing evaluations as successes.
required_paths={
 'PDF text':ROOT/'data'/'extracted_full_text.txt',
 'Token chunks':ROOT/'data'/'chunks.json',
 'Theme distribution':ROOT/'data'/'thematic_counts.json',
 'Qualitative insights':ROOT/'data'/'qualitative_insights.json',
 'Raw indicator JSON':ROOT/'data'/'core_indicators_raw_llm.json',
 'Final indicator JSON':ROOT/'data'/'core_indicators.json',
 'Executive summary':ROOT/'summaries'/'executive_summary.md',
 'Section summaries':ROOT/'summaries'/'section_summaries.json',
 'LLM judge scores':ROOT/'evaluation'/'judge_scores.json',
 'RAG evaluation':ROOT/'rag'/'rag_answers_and_evaluation.json',
 'Prompt benchmark':ROOT/'evaluation'/'prompt_benchmark.json',
}
qa_rows=[]
for label,path in required_paths.items():
    complete=path.exists() and path.stat().st_size>2
    detail=''
    if complete and path.suffix=='.json':
        try:
            content=json.loads(path.read_text(encoding='utf-8'))
            if isinstance(content,list): detail=f'{len(content)} records'
            elif isinstance(content,dict): detail=f'{len(content)} fields'
        except json.JSONDecodeError:
            complete=False; detail='invalid JSON'
    qa_rows.append({'deliverable':label,'present':bool(complete),'detail':detail})
qa_df=pd.DataFrame(qa_rows)
show_table(qa_df)
figures=sorted((ROOT/'figures').glob('*.png'))
print('High-resolution dashboard figures:',len(figures))
if len(figures)<4: print('ACTION REQUIRED: fewer than four dashboard plots were saved.')
if not qa_df.present.all():
    print('PENDING ITEMS: Run or resume the missing sections before submission.')
else:
    print('All listed artifacts exist. Manually verify claims, citations, scoring and report narrative.')
save_json(ROOT/'evaluation'/'submission_readiness.json',{
    'deliverables':qa_rows,'figure_count':len(figures),
    'ready_for_human_audit':bool(qa_df.present.all() and len(figures)>=4)})

## References and reproducibility notes
1. International Monetary Fund (2026). *Republic of Palau: 2025 Article IV Consultation—Press Release; Staff Report; and Statement by the Executive Director*, IMF Country Report No. 26/22. Source: user-supplied PDF (especially pp. 5, 16, 32).
2. Lewis, P. et al. (2020). Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. *NeurIPS 2020*.
3. Johnson, J., Douze, M., and Jégou, H. (2019). Billion-scale similarity search with GPUs. *IEEE Transactions on Big Data* (FAISS).
4. Reimers, N. and Gurevych, I. (2019). Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks. *EMNLP-IJCNLP*.
5. Google. Gemini API documentation (current interface may change). https://ai.google.dev/gemini-api/docs

> **Final checks:** The student's country/year allocation must match this specific Palau report. The code has been structured and syntax-checked prior to delivery, but API-dependent execution must be completed in your own Colab with your key. Include actual screenshots and findings from your completed run in your own written analysis.